# Lesson 07 - Planning Design Pattern

This notebook demonstrates the **Planning Design Pattern** for AI agents using the Microsoft Agent Framework.
You will learn how to break complex travel requests into structured subtasks, assign them to specialist agents,
and execute the resulting plan — all using structured output powered by Pydantic models.


## Εγκατάσταση


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Ανάλυση Εργασίας

Η ανάλυση εργασίας είναι ο πυρήνας του προτύπου σχεδιασμού προγραμματισμού. Αντί να ζητάμε από έναν μόνο πράκτορα να
χειριστεί ένα πολύπλοκο αίτημα από την αρχή μέχρι το τέλος, διαχωρίζουμε το πρόβλημα σε μικρότερα, καλά καθορισμένα **υπο-εργασίες**.
Κάθε υπο-εργασία ανατίθεται σε έναν ειδικό πράκτορα (π.χ., πτήσεις, ξενοδοχεία, δραστηριότητες) με σαφείς
προτεραιότητες και σειρά εξαρτήσεων.

Αυτή η προσέγγιση προσφέρει πολλά οφέλη:
- **Σαφήνεια**: κάθε υπο-εργασία έχει μία και μόνη ευθύνη.
- **Παράλληλοτητα**: ανεξάρτητες υπο-εργασίες μπορούν να εκτελεστούν ταυτόχρονα.
- **Αξιοπιστία**: οι αποτυχίες περιορίζονται σε μεμονωμένες υπο-εργασίες.
- **Παρακολούθηση προϋπολογισμού**: τα κόστη εκτιμώνται ανά υπο-εργασία και συγκεντρώνονται συνολικά.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Δημιουργία Πράκτορα Σχεδιασμού με Δομημένη Έξοδο

Ο πράκτορας σχεδιασμού λειτουργεί ως **υπεύθυνος υποδοχής**. Δεδομένου ενός αιτήματος ταξιδιού υψηλού επιπέδου,
παράγει ένα δομημένο `TravelPlan` — αποσυνθέτοντας το αίτημα σε υποεργασίες, θέτοντας προτεραιότητες,
και εντοπίζοντας εξαρτήσεις έτσι ώστε ένας θυρωρός ή ένα επίπεδο εκτέλεσης να μπορεί να αναλάβει την εργασία.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Εκτέλεση ενός Σχεδίου με Εξειδικευμένα Εργαλεία

Μόλις ο υπάλληλος υποδοχής παράγει ένα δομημένο σχέδιο, ο **υπάλληλος κονσιέρζ** το εκτελεί.
Κάθε εξειδικευμένο εργαλείο χειρίζεται μία κατηγορία υποεργασιών (πτήσεις, ξενοδοχεία, δραστηριότητες). Ο κονσιέρζ
διατρέχει τις υποεργασίες του σχεδίου με τη σειρά εξαρτήσεων και στέλνει κάθε μία στο
κατάλληλο εργαλείο.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Περίληψη

Σε αυτό το μάθημα μάθατε το **Σχεδιαστικό Πρότυπο Προγραμματισμού** για πράκτορες AI:

1. **Αποσύνθεση Εργασιών** — Ένας πράκτορας προγραμματισμού στη ρεσεψιόν διασπά ένα σύνθετο αίτημα ταξιδιού σε
   δομημένες υποεργασίες χρησιμοποιώντας μοντέλα Pydantic, αναθέτοντας κάθε μία σε έναν ειδικό πράκτορα με προτεραιότητες
   και εξαρτήσεις.
2. **Δομημένη Έξοδος** — Με τη χρήση της `response_format` ο πράκτορας επιστρέφει ένα επικυρωμένο
   αντικείμενο `TravelPlan` αντί για ελεύθερο κείμενο, καθιστώντας την επεξεργασία των επόμενων βημάτων αξιόπιστη.
3. **Εκτέλεση Σχεδίου** — Ένας πράκτορας υποδοχής εκτελεί τις υποεργασίες χρησιμοποιώντας εξειδικευμένα εργαλεία
   (`book_flight`, `reserve_hotel`, `book_activity`) για να πραγματοποιήσει το σχέδιο και να αναφέρει τα αποτελέσματα.

Αυτό το πρότυπο διαχωρίζει *τι να κάνεις* (προγραμματισμός) από *πώς να το κάνεις* (εκτέλεση), κάνοντας τους πράκτορες
πιο αρθρωτούς, ελεγχόμενους και εύκολους στη επέκταση.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
